# Zika - Modelling

In [80]:
%reload_ext autoreload
%autoreload 2
from my_lib import *

In [81]:
z_features = ['EW', 'EW_start_date', 'pop', 'Month']
target = 'cases'

## Train Model on all data

In [82]:
from sklearn.linear_model import PoissonRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, make_column_selector(dtype_include=np.number)),
    ('cat', categorical_pipeline, make_column_selector(dtype_include=['object', 'category']))
])

modelling_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', HistGradientBoostingRegressor(loss='poisson'))
])

In [ ]:
from scipy.stats import loguniform
param_grid = {
    'model__max_iter': randint(50, 150),
    # Controls learning step speed
    'model__learning_rate': loguniform(0.01, 0.1),
    # Limits tree depth to prevent overfitting on rare spikes
    'model__max_depth': randint(3, 6)
}

features_to_use = ['Month', 'dengue']

for location in "ABCDEF":
    print(f"Location {location}")
    df_z = pd.read_feather(f"data/zika_{location}.feather")
    df_d = pd.read_feather(f"data/dengue_{location}.feather")
    df_s = pd.read_feather(f"data/score_{location}.feather")
    df_w = pd.read_feather(f"data/weather_{location}.feather")

    X_train, X_test, y_train, y_test = prepare_mining_data(
        df_z,
        df_s,
        df_d,
        df_w,
        z_features,
        target,
        features_to_use=features_to_use
    )

    best_pipeline = tune_pipeline(X_train, y_train, modelling_pipeline, param_grid, 5)

    # modelling_pipeline.fit(X_train, y_train)
    y_pred = best_pipeline.predict(X_test)

    # cleanup negatives and very low predictions
    y_pred = np.clip(y_pred, 0, None)
    y_pred[y_pred < 0.4] = 0
    
    df_s[target] = y_pred

    df_s.to_csv(f"output/baseline_model_{location}.csv", index=False)

Location A
  -> Best Parameters: {'model__learning_rate': np.float64(0.0121675915615859), 'model__max_depth': 5, 'model__max_iter': 101}
  -> Best Validation MAE: 6.6700
MAE across folds: [ 7.55952281 10.20353002  4.44471223  5.08388322  6.81735895]
Mean Validation MAE: 6.82180144511978
Location B
  -> Best Parameters: {'model__learning_rate': np.float64(0.08382394227783455), 'model__max_depth': 3, 'model__max_iter': 145}
  -> Best Validation MAE: 12.1479
MAE across folds: [24.09715847  9.69308698 20.17264103  3.53213353  3.4334028 ]
Mean Validation MAE: 12.185684562411208
Location C
  -> Best Parameters: {'model__learning_rate': np.float64(0.01144022549733573), 'model__max_depth': 5, 'model__max_iter': 58}
  -> Best Validation MAE: 1.8978
MAE across folds: [0.42287851 1.61052412 1.0038744  2.47402655 4.51814763]
Mean Validation MAE: 2.0058902427415495
Location D
  -> Best Parameters: {'model__learning_rate': np.float64(0.08382394227783455), 'model__max_depth': 3, 'model__max_iter': 14

In [84]:
target_mean = np.mean(y_train)
target_std = np.std(y_train)

print(f"Target Mean: {target_mean:.2f}")
print(f"Target Std Dev: {target_std:.2f}")
print(f"Relative Error: {(10 / target_mean) * 100:.1f}%")

Target Mean: 1.28
Target Std Dev: 3.02
Relative Error: 778.6%


## Build Submission File

In [85]:
def make_sub_file(name="baseline_model"):
    dfs = [pd.read_csv(f"output/{name}_{location}.csv") for location in "ABCDEF"]
    df = pd.concat(dfs, ignore_index=True)
    df['ID'] = df['location'] + df['EW'].astype(str)
    df[['ID', 'cases']].to_csv(f"output/{name}_submission.csv", index=False)

make_sub_file()